In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/Mydrive/recsys/data
%cd /content/drive/MyDrives/recsys

Mounted at /content/drive
mkdir: cannot create directory ‘/content/drive/Mydrive’: Operation not supported
[Errno 2] No such file or directory: '/content/drive/MyDrives/recsys'
/content


In [ ]:
!pip install -q implicit faiss-cpu kaggle

In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle/
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
%cd /content/drive/MyDrive/recsys/data
comp = "h-and-m-personalized-fashion-recommendations"
!kaggle competitions download -c {comp} -f transactions_train.csv
!kaggle competitions download -c {comp} -f articles.csv
!kaggle competitions download -c {comp} -f customers.csv
!unzip -o -q '*.zip' && rm -f *.zip
!ls -lh

/content/drive/MyDrive/recsys/data
100% 584M/584M [00:29<00:00, 20.6MB/s]

100% 4.26M/4.26M [00:01<00:00, 2.75MB/s]

100% 97.9M/97.9M [00:06<00:00, 15.5MB/s]


3 archives were successfully processed.
total 3.8G
-rw------- 1 root root  35M Jan 17  2022 articles.csv
-rw------- 1 root root 6.2M Nov 11 16:03 articles.parquet
-rw------- 1 root root  37M Nov 11 16:53 candidates.pkl
-rw------- 1 root root 198M Jan 17  2022 customers.csv
-rw------- 1 root root 159M Nov 11 16:04 customers.parquet
-rw------- 1 root root  98M Nov 11 16:03 transactions_12w.parquet
-rw------- 1 root root 3.3G Jan 17  2022 transactions_train.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/recsys/data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
src = "/content/"
dst = "/content/drive/MyDrive/recsys/data/"

tx = pd.read_csv(src + "transactions_train.csv",
                 dtype={"article_id": "int32", "sales_channel_id": "int8"},
                 parse_dates=["t_dat"])
tx = tx[tx["t_dat"] >= tx["t_dat"].max() - pd.Timedelta(weeks=12)]
tx.to_parquet(dst + "transactions_12w.parquet")

pd.read_csv(src + "articles.csv").to_parquet(dst + "articles.parquet")
pd.read_csv(src + "customers.csv").to_parquet(dst + "customers.parquet")

print(tx.shape, tx["t_dat"].min(), "to", tx["t_dat"].max())
tx.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/transactions_train.csv'

In [ ]:
print("Purchases:", len(tx))
print("Users:", tx["customer_id"].nunique(), "| Items:", tx["article_id"].nunique())
print("\nPurchases per user:")
print(tx.groupby("customer_id").size().describe())

tx.groupby("t_dat").size().plot(figsize=(12, 3), title="Purchases per day")

articles = pd.read_parquet(dst + "articles.parquet")
top = tx["article_id"].value_counts().head(10).rename("count").reset_index()
top.merge(articles[["article_id", "prod_name", "product_type_name"]], on="article_id")

NameError: name 'tx' is not defined

In [ ]:
test_start = tx["t_dat"].max() - pd.Timedelta(days=6)   # last 7 days = test
train = tx[tx["t_dat"] < test_start]
test  = tx[tx["t_dat"] >= test_start]

gt = test.groupby("customer_id")["article_id"].apply(set).to_dict()  # what each user really bought
print("Train:", train["t_dat"].min().date(), "to", train["t_dat"].max().date())
print("Test :", test["t_dat"].min().date(), "to", test["t_dat"].max().date())
print("Test users:", len(gt))

NameError: name 'tx' is not defined

In [ ]:
import numpy as np

def apk(actual, pred, k=12):
    pred = pred[:k]
    hits, score = 0, 0.0
    for i, p in enumerate(pred):
        if p in actual and p not in pred[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def evaluate(preds, gt, k=12):
    maps = [apk(gt[u], preds.get(u, []), k) for u in gt]
    recs = [len(gt[u] & set(preds.get(u, [])[:k])) / len(gt[u]) for u in gt]
    return round(np.mean(maps), 5), round(np.mean(recs), 5)

In [ ]:
last_week = train[train["t_dat"] >= train["t_dat"].max() - pd.Timedelta(days=6)]
top12 = last_week["article_id"].value_counts().index[:12].tolist()

pop_preds = {u: top12 for u in gt}
print("Popularity  -> MAP@12, Recall@12:", evaluate(pop_preds, gt))

Popularity  -> MAP@12, Recall@12: (np.float64(0.00875), np.float64(0.0255))


In [ ]:
recent = train[train["t_dat"] >= train["t_dat"].max() - pd.Timedelta(weeks=4)]
user_hist = (recent.sort_values("t_dat", ascending=False)
                   .drop_duplicates(["customer_id", "article_id"])
                   .groupby("customer_id").head(12)
                   .groupby("customer_id")["article_id"].agg(list).to_dict())

def fill(items):
    return (items + [a for a in top12 if a not in items])[:12]

ba_preds = {u: fill(user_hist.get(u, [])) for u in gt}
print("Buy-again   -> MAP@12, Recall@12:", evaluate(ba_preds, gt))

Buy-again   -> MAP@12, Recall@12: (np.float64(0.02657), np.float64(0.05365))


In [ ]:
def cand_recall(cands, gt):
    r = [len(gt[u] & set(cands.get(u, []))) / len(gt[u]) for u in gt]
    size = np.mean([len(cands.get(u, [])) for u in gt])
    return f"Recall={np.mean(r):.4f}  avg candidates={size:.0f}"

In [ ]:
hist_all = (train.sort_values("t_dat", ascending=False)
                 .drop_duplicates(["customer_id", "article_id"])
                 .groupby("customer_id").head(50)
                 .groupby("customer_id")["article_id"].agg(list).to_dict())
rep_cands = {u: hist_all.get(u, []) for u in gt}

pop50 = last_week["article_id"].value_counts().index[:50].tolist()
pop_cands = {u: pop50 for u in gt}

print("Repurchase:", cand_recall(rep_cands, gt))
print("Popular-50:", cand_recall(pop_cands, gt))

Repurchase: Recall=0.0375  avg candidates=6
Popular-50: Recall=0.0726  avg candidates=50


In [ ]:
b = recent[["customer_id", "t_dat", "article_id"]].drop_duplicates()
b["basket"] = b.groupby(["customer_id", "t_dat"]).ngroup()
b = b[["basket", "article_id"]]

pairs = b.merge(b, on="basket")
pairs = pairs[pairs["article_id_x"] != pairs["article_id_y"]]
pair_cnt = (pairs.groupby(["article_id_x", "article_id_y"]).size()
                 .reset_index(name="n")
                 .sort_values("n", ascending=False)
                 .groupby("article_id_x").head(10))
neighbors = pair_cnt.groupby("article_id_x")["article_id_y"].agg(list).to_dict()
del pairs

def co_cands_for(items, k=50):
    out = []
    for a in items:
        out += [x for x in neighbors.get(a, []) if x not in out]
    return out[:k]

co_cands = {u: co_cands_for(user_hist.get(u, [])) for u in gt}
print("Bought-together:", cand_recall(co_cands, gt))

Bought-together: Recall=0.0397  avg candidates=15


In [ ]:
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

u_cat = train["customer_id"].astype("category")
i_cat = train["article_id"].astype("category")
uid_map = {u: i for i, u in enumerate(u_cat.cat.categories)}
item_ids = i_cat.cat.categories.values

mat = sp.csr_matrix((np.ones(len(train), dtype=np.float32),
                     (u_cat.cat.codes, i_cat.cat.codes)))

als = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01, random_state=42)
als.fit(mat)

test_users = [u for u in gt if u in uid_map]
idx = np.array([uid_map[u] for u in test_users])
ids, _ = als.recommend(idx, mat[idx], N=50, filter_already_liked_items=False)
als_cands = {u: item_ids[row].tolist() for u, row in zip(test_users, ids)}

print("ALS:", cand_recall(als_cands, gt))

/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

ALS: Recall=0.0400  avg candidates=35


In [ ]:
import pickle

def merge(*lists):
    out = []
    for lst in lists:
        out += [x for x in lst if x not in out]
    return out

cands = {u: merge(rep_cands[u], co_cands[u], als_cands.get(u, []), pop_cands[u]) for u in gt}
print("ALL SOURCES:", cand_recall(cands, gt))

pickle.dump(cands, open(dst + "candidates.pkl", "wb"))

ALL SOURCES: Recall=0.1432  avg candidates=97


In [ ]:
def cand_recall(cands, gt):
    r = [len(gt[u] & set(cands.get(u, []))) / len(gt[u]) for u in gt]
    size = np.mean([len(cands.get(u, [])) for u in gt])
    return f"Recall={np.mean(r):.4f}  avg candidates={size:.0f}"

In [ ]:
hist_all = (train.sort_values("t_dat", ascending=False)
                 .drop_duplicates(["customer_id", "article_id"])
                 .groupby("customer_id").head(50)
                 .groupby("customer_id")["article_id"].agg(list).to_dict())
rep_cands = {u: hist_all.get(u, []) for u in gt}

pop50 = last_week["article_id"].value_counts().index[:50].tolist()
pop_cands = {u: pop50 for u in gt}

print("Repurchase:", cand_recall(rep_cands, gt))
print("Popular-50:", cand_recall(pop_cands, gt))

NameError: name 'train' is not defined

In [ ]:
import pandas as pd, numpy as np, scipy.sparse as sp, pickle, gc
from implicit.als import AlternatingLeastSquares

dst = "/content/drive/MyDrive/recsys/data/"
tx       = pd.read_parquet(dst + "transactions_12w.parquet")
articles = pd.read_parquet(dst + "articles.parquet")

# time split
test_start = tx["t_dat"].max() - pd.Timedelta(days=6)
train = tx[tx["t_dat"] < test_start]
test  = tx[tx["t_dat"] >= test_start]
gt = test.groupby("customer_id")["article_id"].apply(set).to_dict()

# metrics
def apk(actual, pred, k=12):
    pred = pred[:k]; hits, score = 0, 0.0
    for i, p in enumerate(pred):
        if p in actual and p not in pred[:i]:
            hits += 1; score += hits / (i + 1)
    return score / min(len(actual), k)

def evaluate(preds, gt, k=12):
    maps = [apk(gt[u], preds.get(u, []), k) for u in gt]
    recs = [len(gt[u] & set(preds.get(u, [])[:k])) / len(gt[u]) for u in gt]
    return round(np.mean(maps), 5), round(np.mean(recs), 5)

# baselines (needed later for comparison and filling)
last_week = train[train["t_dat"] >= train["t_dat"].max() - pd.Timedelta(days=6)]
top12 = last_week["article_id"].value_counts().index[:12].tolist()
recent = train[train["t_dat"] >= train["t_dat"].max() - pd.Timedelta(weeks=4)]
user_hist = (recent.sort_values("t_dat", ascending=False)
                   .drop_duplicates(["customer_id", "article_id"])
                   .groupby("customer_id").head(12)
                   .groupby("customer_id")["article_id"].agg(list).to_dict())
def fill(items):
    return (items + [a for a in top12 if a not in items])[:12]
ba_preds = {u: fill(user_hist.get(u, [])) for u in gt}

print("Ready. Buy-again baseline:", evaluate(ba_preds, gt))

Ready. Buy-again baseline: (np.float64(0.02657), np.float64(0.05365))


In [ ]:
import pandas as pd, numpy as np, scipy.sparse as sp, pickle, gc
from implicit.als import AlternatingLeastSquares

dst = "/content/drive/MyDrive/recsys/data/"
tx       = pd.read_parquet(dst + "transactions_12w.parquet")
articles = pd.read_parquet(dst + "articles.parquet")

# time split
test_start = tx["t_dat"].max() - pd.Timedelta(days=6)
train = tx[tx["t_dat"] < test_start]
test  = tx[tx["t_dat"] >= test_start]
gt = test.groupby("customer_id")["article_id"].apply(set).to_dict()

# metrics
def apk(actual, pred, k=12):
    pred = pred[:k]; hits, score = 0, 0.0
    for i, p in enumerate(pred):
        if p in actual and p not in pred[:i]:
            hits += 1; score += hits / (i + 1)
    return score / min(len(actual), k)

def evaluate(preds, gt, k=12):
    maps = [apk(gt[u], preds.get(u, []), k) for u in gt]
    recs = [len(gt[u] & set(preds.get(u, [])[:k])) / len(gt[u]) for u in gt]
    return round(np.mean(maps), 5), round(np.mean(recs), 5)

# baselines (needed later for comparison and filling)
last_week = train[train["t_dat"] >= train["t_dat"].max() - pd.Timedelta(days=6)]
top12 = last_week["article_id"].value_counts().index[:12].tolist()
recent = train[train["t_dat"] >= train["t_dat"].max() - pd.Timedelta(weeks=4)]
user_hist = (recent.sort_values("t_dat", ascending=False)
                   .drop_duplicates(["customer_id", "article_id"])
                   .groupby("customer_id").head(12)
                   .groupby("customer_id")["article_id"].agg(list).to_dict())
def fill(items):
    return (items + [a for a in top12 if a not in items])[:12]
ba_preds = {u: fill(user_hist.get(u, [])) for u in gt}

print("Ready. Buy-again baseline:", evaluate(ba_preds, gt))

Ready. Buy-again baseline: (np.float64(0.02657), np.float64(0.05365))


In [ ]:
import gc

def to_df(d, name):
    s = pd.Series(d, dtype=object).explode().dropna()
    return pd.DataFrame({"customer_id": s.index, "article_id": s.values.astype("int32"), name: 1})

def build_candidates(hist, users, n=50):
    end  = hist["t_dat"].max()
    lw   = hist[hist["t_dat"] > end - pd.Timedelta(days=7)]
    rec4 = hist[hist["t_dat"] > end - pd.Timedelta(weeks=4)]

    # 1. repurchase
    h = hist.sort_values("t_dat", ascending=False).drop_duplicates(["customer_id", "article_id"])
    rep = h.groupby("customer_id").head(n).groupby("customer_id")["article_id"].agg(list).to_dict()
    r12 = (rec4.sort_values("t_dat", ascending=False).drop_duplicates(["customer_id", "article_id"])
               .groupby("customer_id").head(12).groupby("customer_id")["article_id"].agg(list).to_dict())
    # 2. popular
    pop = lw["article_id"].value_counts().index[:n].tolist()
    # 3. bought together
    b = rec4[["customer_id", "t_dat", "article_id"]].drop_duplicates()
    b["basket"] = b.groupby(["customer_id", "t_dat"]).ngroup()
    b = b[["basket", "article_id"]]
    p = b.merge(b, on="basket"); p = p[p["article_id_x"] != p["article_id_y"]]
    nb = (p.groupby(["article_id_x", "article_id_y"]).size().reset_index(name="c")
           .sort_values("c", ascending=False).groupby("article_id_x").head(10)
           .groupby("article_id_x")["article_id_y"].agg(list).to_dict())
    del p, b; gc.collect()
    def co(items):
        out = []
        for a in items:
            out += [x for x in nb.get(a, []) if x not in out]
        return out[:n]
    # 4. ALS
    uc = hist["customer_id"].astype("category"); ic = hist["article_id"].astype("category")
    umap = {u: i for i, u in enumerate(uc.cat.categories)}; iids = ic.cat.categories.values
    m = sp.csr_matrix((np.ones(len(hist), dtype=np.float32), (uc.cat.codes, ic.cat.codes)))
    model = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.01, random_state=42)
    model.fit(m, show_progress=False)
    known = [u for u in users if u in umap]
    rows = np.array([umap[u] for u in known])
    ids, _ = model.recommend(rows, m[rows], N=n, filter_already_liked_items=False)
    als = {u: iids[r].tolist() for u, r in zip(known, ids)}

    df = pd.concat([
        to_df({u: rep.get(u, []) for u in users}, "src_rep"),
        to_df({u: co(r12.get(u, [])) for u in users}, "src_co"),
        to_df(als, "src_als"),
        to_df({u: pop for u in users}, "src_pop"),
    ])
    df = df.groupby(["customer_id", "article_id"], as_index=False).max().fillna(0)
    df["n_sources"] = df[["src_rep", "src_co", "src_als", "src_pop"]].sum(axis=1)
    return df

In [ ]:
customers = pd.read_parquet(dst + "customers.parquet")[["customer_id", "age"]]
art = articles[["article_id", "product_type_no", "index_group_no", "colour_group_code",
                "section_no", "garment_group_no"]]

def add_features(df, hist):
    end = hist["t_dat"].max()
    # item features
    it = hist.groupby("article_id").agg(item_cnt=("t_dat", "size"), item_price=("price", "mean"),
                                        item_last=("t_dat", "max"), item_first=("t_dat", "min"))
    it["item_cnt_1w"] = hist[hist["t_dat"] > end - pd.Timedelta(days=7)].groupby("article_id").size()
    it["item_cnt_2w"] = hist[hist["t_dat"] > end - pd.Timedelta(days=14)].groupby("article_id").size()
    it["item_days_since_last"] = (end - it["item_last"]).dt.days
    it["item_age_days"] = (end - it["item_first"]).dt.days
    it = it.drop(columns=["item_last", "item_first"]).reset_index()
    # user features
    us = hist.groupby("customer_id").agg(user_cnt=("t_dat", "size"), user_price=("price", "mean"),
                                         user_last=("t_dat", "max"))
    us["user_days_since_last"] = (end - us["user_last"]).dt.days
    us = us.drop(columns="user_last").reset_index()
    # user-item features
    ui = hist.groupby(["customer_id", "article_id"]).agg(ui_cnt=("t_dat", "size"), ui_last=("t_dat", "max"))
    ui["ui_days_since"] = (end - ui["ui_last"]).dt.days
    ui = ui.drop(columns="ui_last").reset_index()
    # user x product-type affinity
    ut = (hist[["customer_id", "article_id"]].merge(art[["article_id", "product_type_no"]], on="article_id")
              .groupby(["customer_id", "product_type_no"]).size().rename("ut_cnt").reset_index())

    df = (df.merge(it, on="article_id", how="left")
            .merge(us, on="customer_id", how="left")
            .merge(ui, on=["customer_id", "article_id"], how="left")
            .merge(art, on="article_id", how="left")
            .merge(ut, on=["customer_id", "product_type_no"], how="left")
            .merge(customers, on="customer_id", how="left"))
    df["price_ratio"] = df["item_price"] / df["user_price"]
    return df

In [ ]:
label_start = train["t_dat"].max() - pd.Timedelta(days=6)
hist_tr  = train[train["t_dat"] < label_start]
label_wk = train[train["t_dat"] >= label_start]

all_lab_users = label_wk["customer_id"].drop_duplicates()
tr_users = all_lab_users.sample(min(30000, len(all_lab_users)), random_state=42).tolist()

tr = build_candidates(hist_tr, tr_users)
pos = label_wk[["customer_id", "article_id"]].drop_duplicates().assign(label=1)
tr = tr.merge(pos, on=["customer_id", "article_id"], how="left")
tr["label"] = tr["label"].fillna(0).astype(int)

tr = tr[tr.groupby("customer_id")["label"].transform("max") == 1]   # keep users with ≥1 hit
tr = add_features(tr, hist_tr).sort_values("customer_id").reset_index(drop=True)
print(tr.shape, "positive rate:", round(tr["label"].mean(), 4))

/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


(846808, 27) positive rate: 0.0124


In [ ]:
import lightgbm as lgb

feats = [c for c in tr.columns if c not in ["customer_id", "article_id", "label"]]
groups = tr.groupby("customer_id", sort=False).size().values

ranker = lgb.LGBMRanker(objective="lambdarank", n_estimators=300, learning_rate=0.05,
                        num_leaves=63, min_child_samples=50, importance_type="gain", verbose=-1)
ranker.fit(tr[feats], tr["label"], group=groups)

pd.Series(ranker.feature_importances_, index=feats).sort_values(ascending=False).head(15)

,0
ui_days_since,61505.817403
user_days_since_last,33992.472487
price_ratio,22822.010380
age,21681.382222
user_price,18616.946764
user_cnt,18439.692084
item_cnt_1w,18415.551880
item_cnt,16204.722616
item_cnt_2w,13295.283489
item_price,11326.252947


In [ ]:
del tr; gc.collect()
te = build_candidates(train, list(gt.keys()))
te = add_features(te, train)
te["score"] = ranker.predict(te[feats])

top = (te.sort_values(["customer_id", "score"], ascending=[True, False])
         .groupby("customer_id").head(12)
         .groupby("customer_id")["article_id"].agg(list).to_dict())
rank_preds = {u: fill(top.get(u, [])) for u in gt}

print("Buy-again baseline      :", evaluate(ba_preds, gt))
print("Two-stage (LightGBM)    :", evaluate(rank_preds, gt))

ranker.booster_.save_model(dst + "ranker_v1.txt")

Buy-again baseline      : (np.float64(0.02657), np.float64(0.05365))
Two-stage (LightGBM)    : (np.float64(0.02758), np.float64(0.05755))


In [ ]:
AGE_BINS = [0, 25, 35, 45, 55, 120]

def add_features_v2(df, hist):
    df = add_features(df, hist)
    end = hist["t_dat"].max()

    # how popular is this item among people of the user's age group (last week)
    lw = (hist[hist["t_dat"] > end - pd.Timedelta(days=7)][["customer_id", "article_id"]]
            .merge(customers, on="customer_id", how="left"))
    lw["age_bin"] = pd.cut(lw["age"], AGE_BINS, labels=False)
    ai = lw.groupby(["age_bin", "article_id"]).size().rename("age_item_cnt_1w").reset_index()
    df["age_bin"] = pd.cut(df["age"], AGE_BINS, labels=False)
    df = df.merge(ai, on=["age_bin", "article_id"], how="left")
    df["age_item_share"] = df["age_item_cnt_1w"] / df["item_cnt_1w"]

    # how often people re-buy this item
    rr = hist.groupby(["customer_id", "article_id"]).size().reset_index(name="c")
    rep_rate = (rr.assign(rep=rr["c"] > 1).groupby("article_id")["rep"].mean()
                  .rename("item_repurchase_rate").reset_index())
    df = df.merge(rep_rate, on="article_id", how="left")

    # is the item trending up or down
    df["item_trend"] = df["item_cnt_1w"] / (df["item_cnt_2w"] - df["item_cnt_1w"] + 1)
    return df

In [ ]:
del te; gc.collect()   # free memory from the v1 test run

tr = build_candidates(hist_tr, tr_users)
tr = tr.merge(pos, on=["customer_id", "article_id"], how="left")
tr["label"] = tr["label"].fillna(0).astype(int)
tr = tr[tr.groupby("customer_id")["label"].transform("max") == 1]
tr = add_features_v2(tr, hist_tr).sort_values("customer_id").reset_index(drop=True)
print(tr.shape, "positive rate:", round(tr["label"].mean(), 4))

(846808, 32) positive rate: 0.0124


In [ ]:
feats = [c for c in tr.columns if c not in ["customer_id", "article_id", "label"]]
groups = tr.groupby("customer_id", sort=False).size().values
cat_cols = ["product_type_no", "index_group_no", "colour_group_code", "section_no", "garment_group_no"]

ranker = lgb.LGBMRanker(objective="lambdarank", n_estimators=300, learning_rate=0.05,
                        num_leaves=63, min_child_samples=50, importance_type="gain", verbose=-1)
ranker.fit(tr[feats], tr["label"], group=groups, categorical_feature=cat_cols)

pd.Series(ranker.feature_importances_, index=feats).sort_values(ascending=False).head(15)

,0
ui_days_since,58372.775010
user_days_since_last,32871.035709
colour_group_code,21181.531567
product_type_no,20902.226087
price_ratio,18796.578943
section_no,16748.776769
user_price,15590.056667
age_item_share,15433.086460
user_cnt,14770.859808
age,14487.881135


In [ ]:

te = build_candidates(train, list(gt.keys()))
te = add_features_v2(te, train)
te["score"] = ranker.predict(te[feats])

top = (te.sort_values(["customer_id", "score"], ascending=[True, False])
         .groupby("customer_id").head(12)
         .groupby("customer_id")["article_id"].agg(list).to_dict())
rank_preds_v2 = {u: fill(top.get(u, [])) for u in gt}

print("Buy-again baseline :", evaluate(ba_preds, gt))
print("Two-stage v1       : (0.02758, 0.05755)")
print("Two-stage v2       :", evaluate(rank_preds_v2, gt))

ranker.booster_.save_model(dst + "ranker_v2.txt")

Buy-again baseline : (np.float64(0.02657), np.float64(0.05365))
Two-stage v1       : (0.02758, 0.05755)
Two-stage v2       : (np.float64(0.02855), np.float64(0.05985))


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, faiss

u_cat = pd.Categorical(train["customer_id"]); i_cat = pd.Categorical(train["article_id"])
user_index, item_index = pd.Index(u_cat.categories), pd.Index(i_cat.categories)
U_t = torch.from_numpy(u_cat.codes.astype(np.int64))
I_t = torch.from_numpy(i_cat.codes.astype(np.int64))

# user feature: age group (5 = unknown)
ub = pd.DataFrame({"customer_id": user_index}).merge(customers, on="customer_id", how="left")
user_age = torch.tensor(pd.cut(ub["age"], AGE_BINS, labels=False).fillna(5).astype(int).values)

# item features: product type, section, garment group, colour, index group
ia = pd.DataFrame({"article_id": item_index}).merge(articles, on="article_id", how="left")
item_cols = ["product_type_no", "section_no", "garment_group_no", "colour_group_code", "index_group_no"]
item_feats = torch.tensor(np.stack([pd.factorize(ia[c])[0] + 1 for c in item_cols], axis=1))
item_card = [int(item_feats[:, j].max()) + 1 for j in range(len(item_cols))]

# popularity correction (stops the model from only learning best-sellers)
cnt = np.bincount(I_t.numpy(), minlength=len(item_index))
logq = torch.tensor(np.log(cnt / cnt.sum()), dtype=torch.float32)

print("users:", len(user_index), "items:", len(item_index), "interactions:", len(U_t))

users: 472757 items: 40058 interactions: 3207805


In [35]:
class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, item_card, d=64):
        super().__init__()
        self.u_emb, self.age_emb = nn.Embedding(n_users, d), nn.Embedding(6, 16)
        self.i_emb = nn.Embedding(n_items, d)
        self.f_embs = nn.ModuleList([nn.Embedding(c, 16) for c in item_card])
        self.u_mlp = nn.Sequential(nn.Linear(d + 16, 128), nn.ReLU(), nn.Linear(128, d))
        self.i_mlp = nn.Sequential(nn.Linear(d + 16 * len(item_card), 128), nn.ReLU(), nn.Linear(128, d))

    def user(self, u):
        x = torch.cat([self.u_emb(u), self.age_emb(user_age[u])], 1)
        return F.normalize(self.u_mlp(x), dim=1)

    def item(self, i):
        f = item_feats[i]
        x = torch.cat([self.i_emb(i)] + [e(f[:, j]) for j, e in enumerate(self.f_embs)], 1)
        return F.normalize(self.i_mlp(x), dim=1)

torch.manual_seed(42)
model = TwoTower(len(user_index), len(item_index), item_card)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
B, tau = 4096, 0.05

for epoch in range(3):
    model.train(); perm = torch.randperm(len(U_t)); total, steps = 0, 0
    for k in range(0, len(perm), B):
        idx = perm[k:k + B]; u, i = U_t[idx], I_t[idx]
        logits = model.user(u) @ model.item(i).T / tau - logq[i][None, :]
        loss = F.cross_entropy(logits, torch.arange(len(idx)))   # in-batch negatives
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item(); steps += 1
    print(f"epoch {epoch + 1}: loss {total / steps:.4f}")

torch.save(model.state_dict(), dst + "two_tower_v1.pt")

epoch 1: loss 8.6350


KeyboardInterrupt: 

In [36]:
import gc
for v in ["model", "opt"]:
    if v in globals(): del globals()[v]
gc.collect()

SEQ = 20
def make_hist(df):
    d = (df.sort_values("t_dat", ascending=False)
           .drop_duplicates(["customer_id", "article_id"])
           .groupby("customer_id").head(SEQ).copy())
    d["i"] = item_index.get_indexer(d["article_id"]) + 1       # 0 = padding
    d["pos"] = d.groupby("customer_id").cumcount()
    users = pd.Index(d["customer_id"].unique())
    H = np.zeros((len(users), SEQ), dtype=np.int64)
    H[users.get_indexer(d["customer_id"]), d["pos"].values] = d["i"].values
    age = user_age[user_index.get_indexer(users)]
    return users, torch.from_numpy(H), age

# history = before the last 2 weeks of train, targets = last 2 weeks of train
split = train["t_dat"].max() - pd.Timedelta(days=13)
h_users, H_tr, age_tr = make_hist(train[train["t_dat"] < split])
tgt = train[train["t_dat"] >= split][["customer_id", "article_id"]].drop_duplicates()
tgt = tgt[tgt["customer_id"].isin(h_users)]
tu = torch.from_numpy(h_users.get_indexer(tgt["customer_id"]).astype(np.int64))
ti = torch.from_numpy(item_index.get_indexer(tgt["article_id"]).astype(np.int64))
print("training pairs:", len(tu))

class TwoTowerV2(nn.Module):
    def __init__(self, n_items, item_card, d=64):
        super().__init__()
        self.h_emb = nn.Embedding(n_items + 1, d, padding_idx=0)   # history items
        self.i_emb = nn.Embedding(n_items + 1, d, padding_idx=0)   # candidate items
        self.age_emb = nn.Embedding(6, 16)
        self.f_embs = nn.ModuleList([nn.Embedding(c, 16) for c in item_card])
        self.u_mlp = nn.Sequential(nn.Linear(d + 16, 128), nn.ReLU(), nn.Linear(128, d))
        self.i_mlp = nn.Sequential(nn.Linear(d + 16 * len(item_card), 128), nn.ReLU(), nn.Linear(128, d))

    def user(self, H, age):
        m = (H > 0).float().unsqueeze(-1)
        h = (self.h_emb(H) * m).sum(1) / m.sum(1).clamp(min=1)    # average of history items
        return F.normalize(self.u_mlp(torch.cat([h, self.age_emb(age)], 1)), dim=1)

    def item(self, i):
        f = item_feats[i]
        x = torch.cat([self.i_emb(i + 1)] + [e(f[:, j]) for j, e in enumerate(self.f_embs)], 1)
        return F.normalize(self.i_mlp(x), dim=1)

torch.manual_seed(42)
model = TwoTowerV2(len(item_index), item_card)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
B, tau = 1024, 0.05

for epoch in range(5):
    model.train(); perm = torch.randperm(len(tu)); total, steps = 0, 0
    for k in range(0, len(perm), B):
        idx = perm[k:k + B]; u, i = tu[idx], ti[idx]
        logits = model.user(H_tr[u], age_tr[u]) @ model.item(i).T / tau - logq[i][None, :]
        loss = F.cross_entropy(logits, torch.arange(len(idx)))
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item(); steps += 1
    print(f"epoch {epoch + 1}: loss {total / steps:.4f}")

torch.save(model.state_dict(), dst + "two_tower_v2.pt")

training pairs: 297939
epoch 1: loss 7.7198
epoch 2: loss 7.2493
epoch 3: loss 7.0298
epoch 4: loss 6.8980
epoch 5: loss 6.7870


In [37]:
def cand_recall(cands, gt):
    r = [len(gt[u] & set(cands.get(u, []))) / len(gt[u]) for u in gt]
    size = np.mean([len(cands.get(u, [])) for u in gt])
    return f"Recall={np.mean(r):.4f}  avg candidates={size:.0f}"

te_users, H_te, age_te = make_hist(train)          # history = all of train
test_u = np.array(list(gt.keys())); ix = te_users.get_indexer(test_u); mask = ix >= 0

model.eval()
with torch.no_grad():
    all_i = torch.arange(len(item_index))
    item_vecs = torch.cat([model.item(all_i[k:k + 50000]) for k in range(0, len(all_i), 50000)]).numpy()
    sel = torch.from_numpy(ix[mask])
    user_vecs = torch.cat([model.user(H_te[sel[k:k + 20000]], age_te[sel[k:k + 20000]])
                           for k in range(0, len(sel), 20000)]).numpy()

index = faiss.IndexFlatIP(64)
index.add(item_vecs.astype("float32"))
_, nn_ids = index.search(user_vecs.astype("float32"), 50)
tt_cands = {u: item_index[row].tolist() for u, row in zip(test_u[mask], nn_ids)}
np.save(dst + "item_vecs_v2.npy", item_vecs)

als_c = te[te["src_als"] == 1].groupby("customer_id")["article_id"].agg(list).to_dict()
cur   = te.groupby("customer_id")["article_id"].agg(list).to_dict()
both  = {u: list(set(cur.get(u, [])) | set(tt_cands.get(u, []))) for u in gt}

print("ALS (50)               :", cand_recall(als_c, gt))
print("Two-tower (50)         :", cand_recall(tt_cands, gt))
print("Current 4 sources      :", cand_recall(cur, gt))
print("4 sources + two-tower  :", cand_recall(both, gt))

ALS (50)               : Recall=0.0400  avg candidates=35
Two-tower (50)         : Recall=0.0291  avg candidates=35
Current 4 sources      : Recall=0.1429  avg candidates=97
4 sources + two-tower  : Recall=0.1523  avg candidates=121


In [38]:
def train_two_tower(hist, epochs=5, seed=42):
    split = hist["t_dat"].max() - pd.Timedelta(days=13)
    hu, H, age = make_hist(hist[hist["t_dat"] < split])
    tg = hist[hist["t_dat"] >= split][["customer_id", "article_id"]].drop_duplicates()
    tg = tg[tg["customer_id"].isin(hu)]
    tu_ = torch.from_numpy(hu.get_indexer(tg["customer_id"]).astype(np.int64))
    ti_ = torch.from_numpy(item_index.get_indexer(tg["article_id"]).astype(np.int64))
    c = np.bincount(item_index.get_indexer(hist["article_id"]), minlength=len(item_index)) + 1
    lq = torch.tensor(np.log(c / c.sum()), dtype=torch.float32)

    torch.manual_seed(seed)
    m = TwoTowerV2(len(item_index), item_card)
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(tu_)); tot, st = 0, 0
        for k in range(0, len(perm), 1024):
            idx = perm[k:k + 1024]; u, i = tu_[idx], ti_[idx]
            logits = m.user(H[u], age[u]) @ m.item(i).T / 0.05 - lq[i][None, :]
            loss = F.cross_entropy(logits, torch.arange(len(idx)))
            o.zero_grad(); loss.backward(); o.step(); tot += loss.item(); st += 1
        print(f"  two-tower epoch {ep + 1}: loss {tot / st:.4f}")
    m.eval()
    return m

def add_tt(df, m, hist, n=50):
    users = df["customer_id"].unique()
    hu, H, age = make_hist(hist)
    ix = hu.get_indexer(users); mask = ix >= 0
    sel = torch.from_numpy(ix[mask]); known = users[mask]
    with torch.no_grad():
        all_i = torch.arange(len(item_index))
        iv = torch.cat([m.item(all_i[k:k + 50000]) for k in range(0, len(all_i), 50000)]).numpy().astype("float32")
        uv = torch.cat([m.user(H[sel[k:k + 20000]], age[sel[k:k + 20000]])
                        for k in range(0, len(sel), 20000)]).numpy().astype("float32")
    fi = faiss.IndexFlatIP(iv.shape[1]); fi.add(iv)
    _, nn_ids = fi.search(uv, n)

    # 5th candidate source
    tt = to_df({u: item_index[r].tolist() for u, r in zip(known, nn_ids)}, "src_tt")
    df = df.merge(tt, on=["customer_id", "article_id"], how="outer")
    src = ["src_rep", "src_co", "src_als", "src_pop", "src_tt"]
    df[src] = df[src].fillna(0)
    df["n_sources"] = df[src].sum(axis=1)

    # tt_score feature = similarity between user vector and item vector
    upos = pd.Index(known).get_indexer(df["customer_id"])
    ipos = item_index.get_indexer(df["article_id"])
    score = np.full(len(df), np.nan, dtype=np.float32)
    ok = np.where((upos >= 0) & (ipos >= 0))[0]
    for k in range(0, len(ok), 1_000_000):
        r = ok[k:k + 1_000_000]
        score[r] = (uv[upos[r]] * iv[ipos[r]]).sum(1)
    df["tt_score"] = score
    return df

In [39]:
if "te" in globals(): del te
gc.collect()

print("Training two-tower on the ranker's history period...")
tt_tr = train_two_tower(hist_tr)

tr = build_candidates(hist_tr, tr_users)
tr = add_tt(tr, tt_tr, hist_tr)
tr = tr.merge(pos, on=["customer_id", "article_id"], how="left")
tr["label"] = tr["label"].fillna(0).astype(int)
tr = tr[tr.groupby("customer_id")["label"].transform("max") == 1]
tr = add_features_v2(tr, hist_tr).sort_values("customer_id").reset_index(drop=True)
print(tr.shape, "positive rate:", round(tr["label"].mean(), 4))

feats  = [c for c in tr.columns if c not in ["customer_id", "article_id", "label"]]
groups = tr.groupby("customer_id", sort=False).size().values
cat_cols = ["product_type_no", "index_group_no", "colour_group_code", "section_no", "garment_group_no"]

ranker3 = lgb.LGBMRanker(objective="lambdarank", n_estimators=300, learning_rate=0.05,
                         num_leaves=63, min_child_samples=50, importance_type="gain", verbose=-1)
ranker3.fit(tr[feats], tr["label"], group=groups, categorical_feature=cat_cols)

pd.Series(ranker3.feature_importances_, index=feats).sort_values(ascending=False).head(15)

Training two-tower on the ranker's history period...
  two-tower epoch 1: loss 7.5749
  two-tower epoch 2: loss 7.1513
  two-tower epoch 3: loss 6.9802
  two-tower epoch 4: loss 6.8687
  two-tower epoch 5: loss 6.7713
(1152892, 34) positive rate: 0.0098


,0
ui_days_since,59831.184612
user_days_since_last,37019.599227
product_type_no,22932.384338
colour_group_code,22304.528071
tt_score,21614.584069
price_ratio,18685.216825
item_cnt_1w,17840.197798
section_no,17468.280657
ui_cnt,17334.154938
age_item_cnt_1w,15638.246008


In [40]:
del tr; gc.collect()

te = build_candidates(train, list(gt.keys()))
te = add_tt(te, model, train)        # 'model' = the two-tower you trained in T2b
print("Candidate recall:", cand_recall(te.groupby("customer_id")["article_id"].agg(list).to_dict(), gt))

te = add_features_v2(te, train)
te["score"] = ranker3.predict(te[feats])

top = (te.sort_values(["customer_id", "score"], ascending=[True, False])
         .groupby("customer_id").head(12)
         .groupby("customer_id")["article_id"].agg(list).to_dict())
rank_preds_v3 = {u: fill(top.get(u, [])) for u in gt}

print("Buy-again baseline :", evaluate(ba_preds, gt))
print("Two-stage v1       : (0.02758, 0.05755)")
print("Two-stage v2       : (0.02855, 0.05985)")
print("Two-stage v3       :", evaluate(rank_preds_v3, gt))

ranker3.booster_.save_model(dst + "ranker_v3.txt")

Candidate recall: Recall=0.1523  avg candidates=121
Buy-again baseline : (np.float64(0.02657), np.float64(0.05365))
Two-stage v1       : (0.02758, 0.05755)
Two-stage v2       : (0.02855, 0.05985)
Two-stage v3       : (np.float64(0.02838), np.float64(0.05957))


In [41]:
pickle.dump(rank_preds_v2, open(dst + "preds_v2.pkl", "wb"))
pickle.dump(top12, open(dst + "top12_popular.pkl", "wb"))
print("Saved")

Saved


In [42]:
import os
sd = "/content/drive/MyDrive/recsys/serving_data/"
os.makedirs(sd, exist_ok=True)

pickle.dump(rank_preds_v2, open(sd + "preds_v2.pkl", "wb"))
pickle.dump(top12, open(sd + "top12_popular.pkl", "wb"))
np.save(sd + "item_ids.npy", np.asarray(item_index, dtype=np.int64))
np.save(sd + "item_vecs_v2.npy", item_vecs)
articles[["article_id", "prod_name", "product_type_name", "colour_group_name"]].to_parquet(sd + "articles_small.parquet")
!ls -lh {sd}

total 21M
-rw------- 1 root root 1.6M Nov 12 15:46 articles_small.parquet
-rw------- 1 root root 314K Nov 12 15:46 item_ids.npy
-rw------- 1 root root 9.8M Nov 12 15:46 item_vecs_v2.npy
-rw------- 1 root root 8.7M Nov 12 15:46 preds_v2.pkl
-rw------- 1 root root   76 Nov 12 15:46 top12_popular.pkl


In [43]:
!pip install -q fastapi uvicorn httpx
!mkdir -p /content/drive/MyDrive/recsys/app

In [44]:
%%writefile /content/drive/MyDrive/recsys/app/main.py
import os, time, pickle
import numpy as np, pandas as pd, faiss
from fastapi import FastAPI, HTTPException

DATA = os.getenv("DATA_DIR", "serving_data")

preds    = pickle.load(open(f"{DATA}/preds_v2.pkl", "rb"))
popular  = pickle.load(open(f"{DATA}/top12_popular.pkl", "rb"))
item_ids = np.load(f"{DATA}/item_ids.npy")
item_vecs = np.load(f"{DATA}/item_vecs_v2.npy").astype("float32")
meta = pd.read_parquet(f"{DATA}/articles_small.parquet").set_index("article_id").to_dict("index")

index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)
pos = {int(a): i for i, a in enumerate(item_ids)}

app = FastAPI(title="H&M Two-Stage Recommender", version="2.0")

def describe(ids):
    out = []
    for a in ids:
        m = meta.get(int(a), {})
        out.append({"article_id": int(a), "name": m.get("prod_name"),
                    "type": m.get("product_type_name"), "colour": m.get("colour_group_name")})
    return out

@app.get("/health")
def health():
    return {"status": "ok", "users_with_recs": len(preds), "items_indexed": index.ntotal}

@app.get("/recommend/{customer_id}")
def recommend(customer_id: str, k: int = 12):
    t0 = time.perf_counter()
    if customer_id in preds:
        items, source = preds[customer_id][:k], "two_stage_v2"
    else:
        items, source = popular[:k], "popular_fallback"     # cold-start user
    return {"customer_id": customer_id, "source": source, "items": describe(items),
            "latency_ms": round((time.perf_counter() - t0) * 1000, 2)}

@app.get("/similar/{article_id}")
def similar(article_id: int, k: int = 12):
    if article_id not in pos:
        raise HTTPException(status_code=404, detail="Unknown article_id")
    t0 = time.perf_counter()
    _, nn = index.search(item_vecs[pos[article_id]][None, :], k + 1)
    ids = [int(item_ids[j]) for j in nn[0] if int(item_ids[j]) != article_id][:k]
    return {"article_id": article_id, "query_item": describe([article_id])[0],
            "similar": describe(ids), "latency_ms": round((time.perf_counter() - t0) * 1000, 2)}

Writing /content/drive/MyDrive/recsys/app/main.py


In [45]:
import sys, importlib, json
sys.path.insert(0, "/content/drive/MyDrive/recsys/app")
os.environ["DATA_DIR"] = sd
import main; importlib.reload(main)
from fastapi.testclient import TestClient
client = TestClient(main.app)

uid = next(iter(rank_preds_v2))
print(json.dumps(client.get("/health").json(), indent=2))
print(json.dumps(client.get(f"/recommend/{uid}?k=5").json(), indent=2))
print(json.dumps(client.get("/recommend/new_user_123?k=3").json(), indent=2))
print(json.dumps(client.get(f"/similar/{top12[0]}?k=5").json(), indent=2))

{
  "status": "ok",
  "users_with_recs": 68984,
  "items_indexed": 40058
}
{
  "customer_id": "00039306476aaf41a07fed942884f16b30abfa83a2a8bea972019098d6406793",
  "source": "two_stage_v2",
  "items": [
    {
      "article_id": 850917001,
      "name": "Sadie Shirt",
      "type": "Shirt",
      "colour": "White"
    },
    {
      "article_id": 915529003,
      "name": "Liliana",
      "type": "Sweater",
      "colour": "Black"
    },
    {
      "article_id": 852584001,
      "name": "SUPREME RW tights",
      "type": "Unknown",
      "colour": "Black"
    },
    {
      "article_id": 898694001,
      "name": "Jasba jersey  shacket",
      "type": "Blazer",
      "colour": "Light Beige"
    },
    {
      "article_id": 863583001,
      "name": "Morris",
      "type": "Sweater",
      "colour": "Off White"
    }
  ],
  "latency_ms": 0.03
}
{
  "customer_id": "new_user_123",
  "source": "popular_fallback",
  "items": [
    {
      "article_id": 909370001,
      "name": "FF PI Haley dr

In [46]:
api = "/content/drive/MyDrive/recsys_api"
!mkdir -p {api}
!cp -r /content/drive/MyDrive/recsys/app {api}/
!cp -r /content/drive/MyDrive/recsys/serving_data {api}/
!rm -rf {api}/app/__pycache__
!touch {api}/app/__init__.py

In [47]:
%%writefile /content/drive/MyDrive/recsys_api/requirements.txt
fastapi
uvicorn[standard]
pandas>=2.2
numpy>=2.0
faiss-cpu>=1.8
pyarrow

Writing /content/drive/MyDrive/recsys_api/requirements.txt


In [48]:
%%writefile /content/drive/MyDrive/recsys_api/Dockerfile
FROM python:3.11-slim
WORKDIR /srv
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ app/
COPY serving_data/ serving_data/
ENV DATA_DIR=/srv/serving_data
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing /content/drive/MyDrive/recsys_api/Dockerfile
